<a href="https://colab.research.google.com/github/AI-Biotechnology-Bioinformatics/Drug_Discovery_AI_Course_2026/blob/main/QSAR_Part_4_Regression_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **AI And Biotechnology/Bioinformatics**

## **AI and Drug Discovery Course: QSAR Modeling**
This notebook demonstrates how to build **Random forest Regression model** to predict pIC50 values.

In [ ]:
# Install LazyPredict
!pip install lazypredict


In [ ]:
# Install SHAP
!pip install shap

In [ ]:
# Import libraries
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.utils import shuffle
from lazypredict.Supervised import LazyRegressor
import shap

## **Loading the Dataset**

In [ ]:
df = pd.read_csv('QSAR_dataset.csv')
df.head()

## **Dataset overview:**
- `molecule_chembl_id`: Unique molecule ID
- `bioactivity_class`: Active/inactive class
- `pIC50`: Continuous potency value (4-9)
- `PubchemFP0` to `PubchemFP880`: 881 binary fingerprints

We have 2443 molecules and 881 features plus 3 identifier/target columns.

## **Features (X) and Target (y)**

Features are our PubChem fingerprints. Target is pIC50 for regression.
We'll exclude non-feature columns first.

In [ ]:
# Exclude non-feature columns
non_feature_cols = ['molecule_chembl_id', 'bioactivity_class', 'pIC50']
X = df.drop(columns=non_feature_cols)
print(X.shape)

In [ ]:
# Target variable
y_reg = df['pIC50']
y_reg

## **Feature Selection – Variance Threshold**

Not all 881 fingerprints are informative. Low-variance features (mostly 0 or 1) add noise. We remove them using `VarianceThreshold`.

In [ ]:
# Apply variance threshold
selection = VarianceThreshold(threshold=(0.8*(1-0.8)))  # Threshold = 0.16
X_var = selection.fit_transform(X)

# Extract the correct feature names
selected_mask = selection.get_support()
feature_names = X.columns[selected_mask]

# Convert immediately to DataFrame
X_var = pd.DataFrame(X_var, columns=feature_names)

print('After variance threshold:', X_var.shape)

In [ ]:
881 - 151

**Result:** 151 informative features retained, 730 removed. Reduces noise, speeds up training, and improves generalization.

## **Split Data into Training and Test Set**

In [ ]:
# Split into 80/20 train/test
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_var, y_reg, test_size=0.2, random_state=123
)
print(f'Training set size: {X_train_reg.shape[0]} molecules')
print(f'Testing set size: {X_test_reg.shape[0]} molecules')

## **Build and Train Random Forest Model**

In [ ]:
# Set random seed for reproducibility
np.random.seed(123)

# Create Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=200)

# Train model
rf_model.fit(X_train_reg, y_train_reg)

## **Evaluate Model's Performance on Test Set**

In [ ]:
r2 = rf_model.score(X_test_reg, y_test_reg)
print(f'Random Forest R² score: {r2:.4f}')

## **R² interpretation:**
- **R² < 0.4** → Poor fit.
- **R² < 0.5-0.7** → Moderate/Acceptable.
- **R² > 0.9** → Very good/High fit

Other metrics like **RMSE** or **MAE** can also be used.

## **Make Predictions on test set**

In [ ]:
# Predict on test set
y_pred_reg = rf_model.predict(X_test_reg)

In [ ]:
y_pred_reg

**y_test_reg**       ..............# actual pIC50  
**y_pred_reg**       ..............# predicted pIC50 values from model

## **Visualize Predictions vs Actual Values**



In [ ]:
ax = sns.regplot(x=y_test_reg, y=y_pred_reg, scatter_kws={'alpha':0.4})
ax.set_xlabel('Experimental pIC50', fontsize='large', fontweight='bold')
ax.set_ylabel('Predicted pIC50', fontsize='large', fontweight='bold')
plt.show()

## **Y-Randomization – Validate Model**

In [ ]:
n_iterations = 5
random_r2_scores = []

for i in range(n_iterations):
    y_train_shuffled = shuffle(y_train_reg, random_state=i)
    rf_random = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_random.fit(X_train_reg, y_train_shuffled)
    y_pred_random = rf_random.predict(X_test_reg)
    r2_random = r2_score(y_test_reg, y_pred_random)
    random_r2_scores.append(r2_random)

print('\nY-Randomization Test Results:')
print(f'Mean R² with shuffled Y: {np.mean(random_r2_scores):.4f}')
print(f'Actual RF R²: {r2:.4f}')

### **Interpretation:**
- Actual RF R² = 0.5,   
- Random R² ≈ -0.2 → Model is learning real chemical patterns, not noise.

## **LazyPredict – Compare Multiple Regression Models**

In [ ]:
print('\n' + '='*60)
print('LAZYPREDICT - COMPARING MULTIPLE REGRESSION MODELS')
print('='*60)

# Initialize LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)

# Fit all models
models, predictions = reg.fit(X_train_reg, X_test_reg, y_train_reg, y_test_reg)

# Display results
print(models.head(10))  # Top 10 models

## **Visualize Top Models**

In [ ]:
top_models = models.head(10)

plt.figure(figsize=(10, 6))

# Define distinct colors for each bar
colors = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'magenta', 'lime', 'brown', 'pink']

# Vertical bar plot with multiple colors and alpha for transparency
plt.bar(top_models.index, top_models['R-Squared'], color=colors, alpha=0.7)

plt.ylabel('R² Score', fontsize=12, fontweight='bold')
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.title('Top 10 Models from LazyPredict', fontsize=14, fontweight='bold')

# Rotate x-axis labels for readability
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## **Compare Best Model with Random Forest**

In [ ]:
best_model_name = models.index[0]
best_r2 = models['R-Squared'].iloc[0]
print(f'Best model from LazyPredict: {best_model_name}')
print(f'Best R² score: {best_r2:.4f}')
print(f'Random Forest R²: {r2:.4f}')
print(f'Improvement: {(best_r2 - r2)*100:.2f}%')

# Check Random Forest rank
if 'RandomForestRegressor' in models.index:
    rf_rank = models.index.get_loc('RandomForestRegressor') + 1
    print(f'Random Forest ranked: {rf_rank} out of {len(models)}')

## **Random Forest Feature Importances using SHAP values**

In [ ]:
# Create an explainer and calculate SHAP values for the test set
explainer = shap.Explainer(rf_model)
shap_values = explainer(X_test_reg)

# Summary plot
shap.summary_plot(shap_values, X_test_reg, feature_names= feature_names)